# Imports

In [ ]:
# import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import matplotlib.pyplot as plt  # Correct import for pyplot
from scipy.stats import norm
from matplotlib.ticker import LogFormatterSciNotation
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [ ]:
Path.cwd()

# Helper functions

In [ ]:
# Assuming your DataFrame is named 'df'
# Create a function to calculate the Z-test for a single row
def calculate_z_test(row):
    # Check if both models have data
    if pd.isna(row['REM (mean)']) or pd.isna(row['BHM (mean)']):
        return pd.Series({'Z_score': np.nan, 'P_value': np.nan, 'Significant': np.nan})
    
    # Extract values
    mean_bhm = row['BHM (mean)']
    lower_bhm = row['BHM (lower [95% CI])']
    upper_bhm = row['BHM (upper [95% CI])']
    
    mean_rem = row['REM (mean)']
    lower_rem = row['REM (lower [95% CI])']
    upper_rem = row['REM (upper [95% CI])']
    
    # The pooled BHM/REM estimates are geometric means and their intervals are
    # back-transformed from symmetric log-scale intervals, so the comparison
    # must be done on the log scale (arithmetic-scale SEs understate/overstate
    # the true uncertainty for a lognormal-shaped estimate).
    se_bhm = (np.log(upper_bhm) - np.log(lower_bhm)) / (2 * 1.959964)
    se_rem = (np.log(upper_rem) - np.log(lower_rem)) / (2 * 1.959964)
    
    # Calculate SE of the difference
    se_diff = np.sqrt(se_bhm**2 + se_rem**2)
    
    # Calculate Z-score on the log scale
    z_score = (np.log(mean_bhm) - np.log(mean_rem)) / se_diff
    
    # Calculate two-tailed p-value
    p_value = 2 * (1 - norm.cdf(abs(z_score)))
    
    # Determine significance (using alpha = 0.05)
    significant = p_value < 0.05

    # Add effect size calculation to the function
    effect_size = z_score * np.sqrt(2)

    return pd.Series({'Z_score': z_score, 'P_value': p_value, 'Significant': significant, 'Effect_Size': effect_size})


In [ ]:
def plot_bhm_rem_comparison(data, figsize=(12, 8), rotation=45, fontsize=14, 
                           bhm_color='blue', rem_color='red', log_scale=True, 
                           parameter_subscripts=None, capsize=7, markersize_rem=8, 
                           markersize_bhm=8):
    """
    Plot comparison of BHM and REM estimates with error bars.
    
    Parameters:
    -----------
    data : pandas DataFrame
        DataFrame containing BHM and REM estimates with confidence intervals
    figsize : tuple, optional
        Figure size (width, height) in inches
    rotation : int, optional
        Rotation angle for x-axis labels
    fontsize : int, optional
        Font size for x-axis labels
    bhm_color : str, optional
        Color for BHM estimates
    rem_color : str, optional
        Color for REM estimates
    log_scale : bool, optional
        Whether to use log scale for y-axis
    parameter_subscripts : dict, optional
        Dictionary mapping parameter names to LaTeX formatted strings with subscripts
        
    Returns:
    --------
    fig, ax : matplotlib Figure and Axes objects
    """
    # Set up the plot
    fig, ax = plt.subplots(figsize=figsize)
    
    # Set positions for the groups
    n_params = len(data)
    x_pos = np.arange(n_params)
    
    # Width of the bars
    width = 0.35
    
    # Plot BHM estimates with error bars
    bhm_means = data['BHM (mean)']
    bhm_lower_err = bhm_means - data['BHM (lower [95% CI])']
    bhm_upper_err = data['BHM (upper [95% CI])'] - bhm_means
    bhm_error = [bhm_lower_err.values, bhm_upper_err.values]
    
    ax.errorbar(x_pos - width/2, bhm_means, yerr=bhm_error, 
                fmt='d', color=bhm_color, label='BHM', capsize=capsize, markersize=markersize_bhm)

    # Plot REM estimates with error bars
    rem_means = data['REM (mean)']
    rem_lower_err = rem_means - data['REM (lower [95% CI])']
    rem_upper_err = data['REM (upper [95% CI])'] - rem_means
    rem_error = [rem_lower_err.values, rem_upper_err.values]
    
    ax.errorbar(x_pos + width/2, rem_means, yerr=rem_error, 
                fmt='s', color=rem_color, label='REM', capsize=capsize, markersize=markersize_rem)

    # Create a list of subscript labels in the same order as your data
    subscript_labels = [parameter_subscripts.get(param, param) for param in data['Parameter']]
    
    # Customize the plot
    ax.set_xlabel('Kinetic Rate', weight='bold')
    ax.set_ylabel('Mean', weight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(subscript_labels, rotation=rotation, ha='right', fontsize=fontsize)
    ax.legend()
    
    # Set log scale if requested
    if log_scale:
        ax.set_yscale('log')
        ax.set_ylabel('Mean (Log Scale)', weight='bold')
        
        # Use LogFormatter to display ticks in scientific notation
        formatter = LogFormatterSciNotation(labelOnlyBase=False)
        ax.yaxis.set_major_formatter(formatter)
        
        # Set evenly spaced y-ticks for log scale
        y_min = min(bhm_means.min(), rem_means.min())
        y_max = max(bhm_means.max(), rem_means.max())
        
        # Create evenly spaced ticks in log space
        log_min = np.floor(np.log10(y_min))
        log_max = np.ceil(np.log10(y_max))
        y_ticks = np.logspace(log_min, log_max, num=int(log_max-log_min)+1)
        
        ax.set_yticks(y_ticks)
    else:
        # For linear scale, use default evenly spaced ticks
        ax.yaxis.set_major_locator(plt.MaxNLocator(nbins='auto', steps=[1, 2, 5, 10]))
    
    plt.tight_layout()
    
    return fig, ax

# Data preparation

In [ ]:


# Get current working directory
script_dir = Path.cwd()  # This gives: .../Meta-analysis-for-OXTR-trafficking/code/python

# Navigate up two levels to the project root, then to data folder
project_root = script_dir.parent.parent
data_path = project_root / 'data' / 'GPCR_parameters.xlsx'

print(f"Looking for file at: {data_path}")

# Read the file
if data_path.exists():
    data = pd.read_excel(data_path)
    print("✅ Data loaded successfully!")
    print(f"Columns: {data.columns.tolist()}")
    print(data.head())
else:
    print(f"❌ File not found! Check path: {data_path}")
    # List contents of data directory for debugging
    data_dir = project_root / 'data'
    if data_dir.exists():
        print(f"Files in data directory: {list(data_dir.iterdir())}")
    else:
        print("Data directory doesn't exist!")

In [ ]:
data

# Geometric mean analysis

## BHM v REM mean & CI comparison

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Bring in the data
data_2 = pd.read_excel(data_path, sheet_name='GPCR_parameters_beta12_geomean')

# Create a figure with two subplots, but now with the larger values on top
f, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(13, 8))
f.subplots_adjust(hspace=0.05)  # Adjust space between axes

# Set positions for the groups
n_params = len(data_2)
x_pos = np.arange(n_params)
width = 0.35

# Plot data on both axes
for ax in [ax1, ax2]:
    # Plot BHM estimates with error bars
    bhm_means = data_2['BHM (mean)']
    bhm_lower_err = bhm_means - data_2['BHM (lower [95% CI])']
    bhm_upper_err = data_2['BHM (upper [95% CI])'] - bhm_means
    bhm_error = [bhm_lower_err.values, bhm_upper_err.values]
    ax.errorbar(x_pos - width/2, bhm_means, yerr=bhm_error, 
                fmt='d', color='blue', label='BHM', capsize=7, markersize=8)

    # Plot REM estimates with error bars
    rem_means = data_2['REM (mean)']
    rem_lower_err = rem_means - data_2['REM (lower [95% CI])']
    rem_upper_err = data_2['REM (upper [95% CI])'] - rem_means
    rem_error = [rem_lower_err.values, rem_upper_err.values]
    ax.errorbar(x_pos + width/2, rem_means, yerr=rem_error, 
                fmt='s', color='red', label='REM', capsize=7, markersize=6)

    # Set log scale
    ax.set_yscale('log')

# Now set the ylims correctly - top plot shows higher values, bottom shows lower values
ax1.set_ylim(1e4, 1e7)   # Upper part of the data (higher values)
ax2.set_ylim(0, 1e4)  # Lower part of the data (lower values)

# Hide the spines between the axes
ax1.spines['bottom'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax1.xaxis.tick_top()
ax1.tick_params(labeltop=False)  # Don't put tick labels at the top
ax2.xaxis.tick_bottom()

# Add diagonal lines to indicate the break - now at the bottom of the top plot
d = 0.010  # Size of the diagonal lines
kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
ax1.plot((-d, +d), (-d, +d), **kwargs)        # Top-left diagonal
ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)  # Top-right diagonal

kwargs.update(transform=ax2.transAxes)
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # Bottom-left diagonal
ax2.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)  # Bottom-right diagonal

# Simple mapping to add subscripts where needed
parameter_subscripts = {
    "konG": r"$k_{onG}$",
    "koffG": r"$k_{offG}$",
    "kact": r"$k_{act}$",
    "kinact": r"$k_{inact}$",
    "kP": r"$k_P$",
    "kdp": r"$k_{dp}$",
    "kba 1": r"$k_{ba1}$",
    "kba 2": r"$k_{ba2}$",
    "kcons_int": r"$k_{cons\_int}$",
    "kint": r"$k_{int}$",
    "krec": r"$k_{rec}$",
    "kexpt": r"$k_{expt}$",
    "krgt": r"$k_{rgt}$",
    "kdeg": r"$k_{deg}$"
}

# Create a list of subscript labels in the same order as your data
subscript_labels = [parameter_subscripts[param] for param in data_2['Parameter']]

# Customize the plot
ax2.set_xlabel('Kinetic Rate', weight='bold')
f.text(0.005, 0.5, 'Mean (Log Scale)', va='center', rotation='vertical', weight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(subscript_labels, rotation=0, ha='right', fontsize=14)

# Add a single legend for both subplots
handles, labels = ax1.get_legend_handles_labels()
f.legend(handles, labels, loc='upper right', fontsize=15)

# Invert the order of the subplots so higher values appear above lower values
ax1.set_position([0.125, 0.5, 0.775, 0.4])  # [left, bottom, width, height]
ax2.set_position([0.125, 0.1, 0.775, 0.4])

plt.tight_layout()
plt.show()

## statistical analysis of BHM vs REM means for geometric mean

### analysis

In [ ]:
# call the geometric mean data
data = pd.read_excel(data_path, sheet_name='GPCR_parameters_geomean')

# Apply the function to each row
results = data.apply(calculate_z_test, axis=1)

# Combine the results with the original DataFrame
df_results = pd.concat([data, results], axis=1)

# Display the results
print(df_results[['Parameter', 'BHM (mean)', 'REM (mean)', 
                  'Z_score', 'P_value', 'Significant']])

# combined the new results with the parameter names 

comparative_results = df_results[['Parameter', 'BHM (mean)', 'REM (mean)',
                  'Z_score', 'P_value', 'Significant', 'Effect_Size']]

# Apply scientific notation with 3 significant figures to numeric columns
numeric_columns = ['BHM (mean)', 'REM (mean)', 'Z_score', 'P_value', 'Effect_Size']
formatters = {col: lambda x: f"{x:.2e}" for col in numeric_columns}

# Display with formatting
display(comparative_results.style.format(formatters))
comparative_results_all = comparative_results



### Plot z score

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.patches as mpatches

# Filter out rows with NaN in REM for comparison
comparison_df = comparative_results.dropna(subset=['REM (mean)']).copy()

# Create shortened parameter names for display
def shorten_parameter(name):
    # Custom shortening rules - adjust based on your parameter names
    short_names = {
        'Kd-OXTR: G-protein': 'Kd-OXTR',
        'kact': 'kact',
        'kinact': 'kinact',
        'kP': 'kP', 
        'kdp': 'kdp',
        'kkba': 'kkba',
        'kint': 'kint',
        'kcons_int': 'kcons_int',
        'krecEE': 'krecEE',
        'kexpt': 'kexpt',
        'krgt': 'krgt',
        'kdeg': 'kdeg'
    }
    return short_names.get(name, name[:10] + '...' if len(name) > 10 else name)

# Apply shortening
comparison_df['Parameter_Short'] = comparison_df['Parameter'].apply(shorten_parameter)

plt.figure(figsize=(14, 8))

colors = ["#F51D01" if sig else '#4D64A3' for sig in comparison_df['Significant']]

# Use shortened names for plotting
bars = plt.barh(comparison_df['Parameter_Short'], comparison_df['Effect_Size'], 
                color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)

plt.axvline(x=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
plt.ylabel('Parameters', fontsize=12, fontweight='bold')
plt.xlabel('Effect Size', fontsize=12, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')

# Add effect size values
for bar, effect_size in zip(bars, comparison_df['Effect_Size']):
    width = bar.get_width()
    y_pos = bar.get_y() + bar.get_height()/2
    
    if width >= 0:
        x_pos = width + 0.1
        ha = 'left'
    else:
        x_pos = width - 0.0001
        ha = 'right'
    
    plt.text(x_pos, y_pos, 
             f'{effect_size:.2f}', ha=ha, va='center', 
             fontsize=9, fontweight='bold', color='black')

# Create legend (using original parameter names)
legend_handles = []
legend_handles.append(mpatches.Patch(color="#F51D01", label='Significant (p < 0.05)'))

sig_results = comparison_df[comparison_df['Significant']]
if not sig_results.empty:
    for _, row in sig_results.iterrows():
        legend_handles.append(
            mpatches.Patch(color='white', 
                          label=f"{row['Parameter']}: p={row['P_value']:.1e}",  # Use full name in legend
                          alpha=0)
        )

legend_handles.append(mpatches.Patch(color='#4D64A3', label='Not Significant'))

plt.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1, 0.5))

# Add statistics
sig_count = comparison_df['Significant'].sum()
total_count = len(comparison_df)
plt.figtext(0.02, 0.02, f'Significant parameters: {sig_count}/{total_count} ({sig_count/total_count*100:.1f}%)', 
            fontsize=10, style='italic')

# Add explanatory note
plt.figtext(0.5, -0.05, 
            'Effect Sizes: BHM vs REM Parameter Comparison\n(Red bars indicate statistically significant differences)',
            ha='center', fontsize=10, style='italic')

plt.subplots_adjust(left=0.25, right=0.75)
plt.tight_layout()
plt.show()

# Beta arrestin sub analysis for geometric mean

## Prepare data for kba 1 and kba 2

In [ ]:
data_kba_bhm = pd.read_excel(data_path, sheet_name='beta_arrestin_bhm')
data_kba_bhm

In [ ]:
data_kba_rem = pd.read_excel(data_path, sheet_name='beta_arrestin_rem_geomean')
data_kba_rem

In [ ]:
df_kba = data_kba_bhm[['mean','HDI(2.5%) Minimum', 'HDI(97.5%) Maximum']]
# replace 'mean', 'HDI(2.5%) Minimum', and 'HDI(97.5%) Maximum' with the BHM (mean)	BHM (lower [95% CI])	BHM (upper [95% CI])
df_kba.columns = ['BHM (mean)', 'BHM (lower [95% CI])', 'BHM (upper [95% CI])']
# drop the last row
df_kba = df_kba[:-1]
# add a parameter column as the first column
df_kba.insert(0, 'Parameter', ['kba 1','kba 2'])
# add the REM rows for kba
display(df_kba)
# append the REM kba values to the df_kba dataframe
kba1_2_data =pd.concat([df_kba, data_kba_rem[['REM (mean)', 'REM (lower [95% CI])', 'REM (upper [95% CI])']]], axis=1)
display(kba1_2_data)

In [ ]:
# read in geometric mean data
data = pd.read_excel(data_path, sheet_name='GPCR_parameters_geomean')
# extract the bhm AND REM kba values from the original data
cumulative_kba = data[data['Parameter'].str.contains('kba')]
display(cumulative_kba)

In [ ]:
all_kba =pd.concat([cumulative_kba, kba1_2_data], axis=0)
display(all_kba)

## Statistical analysis of kba 1 and kba 2

In [ ]:
# Apply the function to each row
results = all_kba.apply(calculate_z_test, axis=1)

# Combine the results with the original DataFrame
df_results = pd.concat([all_kba, results], axis=1)

# Display the results
print(df_results[['Parameter', 'BHM (mean)', 'REM (mean)', 
                  'Z_score', 'P_value', 'Significant']])
# combined the new results with the parameter names 

comparative_results = df_results[['Parameter', 'BHM (mean)', 'REM (mean)',
                  'Z_score', 'P_value', 'Significant', 'Effect_Size']]

# Apply scientific notation with 3 significant figures to numeric columns
numeric_columns = ['BHM (mean)', 'REM (mean)', 'Z_score', 'P_value', 'Effect_Size']
formatters = {col: lambda x: f"{x:.2e}" for col in numeric_columns}

# Display with formatting
display(comparative_results.style.format(formatters))

### Plot of Z score

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib.patches as mpatches

# Filter out rows with NaN in REM for comparison
comparison_df = comparative_results.dropna(subset=['REM (mean)']).copy()

# Create shortened parameter names for display
def shorten_parameter(name):
    # Custom shortening rules - adjust based on your parameter names
    short_names = {
        'Kd-OXTR: G-protein': 'Kd-OXTR',
        'kact': 'kact',
        'kinact': 'kinact',
        'kP': 'kP', 
        'kdp': 'kdp',
        'kkba': 'kkba',
        'kint': 'kint',
        'kcons_int': 'kcons_int',
        'krecEE': 'krecEE',
        'kexpt': 'kexpt',
        'krgt': 'krgt',
        'kdeg': 'kdeg'
    }
    return short_names.get(name, name[:10] + '...' if len(name) > 10 else name)

# Apply shortening
comparison_df['Parameter_Short'] = comparison_df['Parameter'].apply(shorten_parameter)

plt.figure(figsize=(14, 8))

colors = ['#FF6B6B' if sig else '#B0B0B0' for sig in comparison_df['Significant']]

# Use shortened names for plotting
bars = plt.barh(comparison_df['Parameter_Short'], comparison_df['Effect_Size'], 
                color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)

plt.axvline(x=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
plt.ylabel('Parameters', fontsize=12, fontweight='bold')
plt.xlabel('Effect Size', fontsize=12, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')

# Add effect size values
for bar, effect_size in zip(bars, comparison_df['Effect_Size']):
    width = bar.get_width()
    y_pos = bar.get_y() + bar.get_height()/2
    
    if width >= 0:
        x_pos = width + 0.1
        ha = 'left'
    else:
        x_pos = width - 0.0001
        ha = 'right'
    
    plt.text(x_pos, y_pos, 
             f'{effect_size:.2f}', ha=ha, va='center', 
             fontsize=9, fontweight='bold', color='black')

# Create legend (using original parameter names)
legend_handles = []
legend_handles.append(mpatches.Patch(color='#FF6B6B', label='Significant (p < 0.05)'))

sig_results = comparison_df[comparison_df['Significant']]
if not sig_results.empty:
    for _, row in sig_results.iterrows():
        legend_handles.append(
            mpatches.Patch(color='white', 
                          label=f"{row['Parameter']}: p={row['P_value']:.1e}",  # Use full name in legend
                          alpha=0)
        )

legend_handles.append(mpatches.Patch(color='#B0B0B0', label='Not Significant'))

plt.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1, 0.5))

# Add statistics
sig_count = comparison_df['Significant'].sum()
total_count = len(comparison_df)
plt.figtext(0.02, 0.02, f'Significant parameters: {sig_count}/{total_count} ({sig_count/total_count*100:.1f}%)', 
            fontsize=10, style='italic')

# Add explanatory note
plt.figtext(0.5, -0.05, 
            'Effect Sizes: BHM vs REM Parameter Comparison\n(Red bars indicate statistically significant differences)',
            ha='center', fontsize=10, style='italic')

plt.subplots_adjust(left=0.25, right=0.75)
plt.tight_layout()
plt.show()

## plot a log box plot for kba 1 and kba 2

In [ ]:

# Parameter name mapping to LaTeX format with subscripts
parameter_subscripts = {
"kba": r"Aggregate $k_{kba}$",
"kba 1": r"$k_{ba1}$",
"kba 2": r"$k_{ba2}$"
    }

# Generate the plot
fig, ax = plot_bhm_rem_comparison(
    all_kba, 
    figsize=(10, 5),
    rotation=0,
    bhm_color='blue',
    rem_color='red',
    parameter_subscripts=parameter_subscripts,
    capsize=10,
    markersize_bhm=10,
    markersize_rem=8
)

# Add a title if needed
#ax.set_title('BHM vs REM Estimates for Parameter kkba 1')
plt.show()

# REM Heterogenity test

In [ ]:
# call the geometric mean data
data = pd.read_excel(data_path, sheet_name='rem_heterogeneity')
data


### Graphical test for heterogenity

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches

# Clean the data first
data_clean = data.copy()

# Fix Parameter column - ensure all values are strings and handle NaN
data_clean['Parameter'] = data_clean['Parameter'].astype(str)
data_clean = data_clean[data_clean['Parameter'] != 'nan']  # Remove NaN strings

# Also ensure I2 column is numeric
data_clean['I2'] = pd.to_numeric(data_clean['I2'], errors='coerce')

# Remove any remaining rows with missing data
data_clean = data_clean.dropna(subset=['Parameter', 'I2'])

print(f"Cleaned data: {len(data_clean)} rows")

# Now create the plot with enhanced styling
plt.figure(figsize=(10, 6))

# Sort by I2 for better visualization
data_sorted = data_clean.sort_values('I2', ascending=True)

# Create color scheme based on significance
colors = ['#EB4731' if sig != 'NS' else '#4D64A3' for sig in data_sorted['Significance']]

# Create horizontal bar plot
bars = plt.barh(data_sorted['Parameter'], data_sorted['I2'], 
                color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)

# Add reference lines for heterogeneity interpretation
plt.axvline(x=50, color='blue', linestyle='--', alpha=0.7, label='50% (Moderate)', linewidth=2)
plt.axvline(x=75, color='red', linestyle='--', alpha=0.7, label='75% (High)', linewidth=2)

# Customize the plot
plt.xlabel('I² (%) - Heterogeneity', fontsize=12, fontweight='bold')
plt.ylabel('Parameters', fontsize=12, fontweight='bold')
plt.xlim(0, 105)
plt.grid(True, alpha=0.3, axis='x')

# Add I² values on bars with better positioning
for bar, I2_value in zip(bars, data_sorted['I2']):
    # Position text inside the bar for better readability
    if I2_value > 10:  # If bar is long enough, put text inside
        text_color = 'black'
        x_pos = I2_value - 5
        ha = 'right'
    else:  # If bar is short, put text outside
        text_color = 'black'
        x_pos = I2_value + 1
        ha = 'left'
    
    plt.text(x_pos, bar.get_y() + bar.get_height()/2, 
             f'{I2_value}%', ha=ha, va='center', 
             fontsize=9, fontweight='bold', color=text_color)

# Add legend with better positioning
plt.legend(loc='lower right', framealpha=0.9)

# Adjust layout to make room for the note
plt.tight_layout(rect=[0, 0.05, 1, 0.95])

# Add explanatory note
plt.figtext(0.5, 0.01, 
            'Heterogeneity Analysis: I² Values by Parameter\n(Red bars indicate statistically significant heterogeneity)',
            ha='center', fontsize=10, style='italic')

plt.show()

In [ ]:
# Add this after your plot to show key statistics
print("\n" + "="*50)
print("HETEROGENEITY SUMMARY")
print("="*50)

total_params = len(data_clean)
sig_het = len(data_clean[data_clean['Significance'] != 'NS'])
extreme_het = len(data_clean[data_clean['I2'] >= 90])
high_het = len(data_clean[(data_clean['I2'] >= 75) & (data_clean['I2'] < 90)])
moderate_het = len(data_clean[(data_clean['I2'] >= 50) & (data_clean['I2'] < 75)])
low_het = len(data_clean[data_clean['I2'] < 50])

print(f"Total parameters analyzed: {total_params}")
print(f"Parameters with significant heterogeneity: {sig_het} ({sig_het/total_params*100:.1f}%)")
print(f"Extreme heterogeneity (I² ≥ 90%): {extreme_het} parameters")
print(f"High heterogeneity (I² 75-89%): {high_het} parameters")
print(f"Moderate heterogeneity (I² 50-74%): {moderate_het} parameters")
print(f"Low heterogeneity (I² < 50%): {low_het} parameters")
print(f"Average I²: {data_clean['I2'].mean():.1f}%")
print(f"Median I²: {data_clean['I2'].median():.1f}%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.patches as mpatches

# Clean the data (same as before)
data_clean = data.copy()
data_clean['Parameter'] = data_clean['Parameter'].astype(str)
data_clean = data_clean[data_clean['Parameter'] != 'nan']
data_clean['tau'] = pd.to_numeric(data_clean['tau'], errors='coerce')
data_clean['Q'] = pd.to_numeric(data_clean['Q'], errors='coerce')
data_clean = data_clean.dropna(subset=['Parameter', 'tau', 'Q'])

print(f"Cleaned data: {len(data_clean)} rows")

# Create subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Colors for significance
colors = ['#FF6B6B' if sig != 'NS' else '#B0B0B0' for sig in data_clean['Significance']]

# Plot 1: Tau values (between-study variance)
data_tau = data_clean.sort_values('tau', ascending=True)
bars1 = ax1.barh(data_tau['Parameter'], data_tau['tau'], 
                 color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax1.set_xlabel('τ (Between-study Variance)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Parameters', fontsize=12, fontweight='bold')
ax1.set_title('Between-study Variance (τ) by Parameter', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# Add tau values on bars
for bar, tau_val in zip(bars1, data_tau['tau']):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
             f'{tau_val:.2f}', ha='left', va='center', 
             fontsize=9, fontweight='bold')

# Plot 2: Q statistics (log scale due to large range)
data_q = data_clean.sort_values('Q', ascending=True)
bars2 = ax2.barh(data_q['Parameter'], data_q['Q'], 
                 color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax2.set_xlabel('Q Statistic (Log Scale)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Parameters', fontsize=12, fontweight='bold')
ax2.set_title("Cochran's Q Statistic by Parameter", fontsize=14, fontweight='bold')
ax2.set_xscale('log')  # Log scale for Q due to large values
ax2.grid(True, alpha=0.3, axis='x')

# Add Q values on bars
for bar, q_val in zip(bars2, data_q['Q']):
    ax2.text(bar.get_width() * 1.1, bar.get_y() + bar.get_height()/2, 
             f'{q_val:.0f}', ha='left', va='center', 
             fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
data_clean

In [ ]:
plt.figure(figsize=(12, 8))

# remove the second row of data_clean
data_clean = data_clean.drop(data_clean.index[1])

# Create scatter plot
scatter = plt.scatter(data_clean['tau'], data_clean['Q'], 
                     s=100, alpha=0.7, 
                     c=['#EB4731' if sig != 'NS' else '#4D64A3' for sig in data_clean['Significance']])

# Add parameter labels
for i, row in data_clean.iterrows():
    plt.annotate(row['Parameter'], 
                (row['tau'], row['Q']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=14, alpha=0.8)

plt.xlabel('Between-study Variance (τ)', fontsize=12, fontweight='bold')
plt.ylabel('Cochran\'s Q Statistic', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)

# Use log scale for Q axis due to large range
plt.yscale('log')

# Add legend
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#EB4731', 
               markersize=10, label='Significant Heterogeneity'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#4D64A3', 
               markersize=10, label='Non-significant Heterogeneity')
]
plt.legend(handles=legend_elements, loc='lower right')

# Add explanatory note
plt.figtext(0.5, -0.05, 
            'Relationship Between τ and Q Statistics\n(Red = Significant Heterogeneity \'I² (%)\')',
            ha='center', fontsize=10, style='italic')


plt.tight_layout()
plt.show()

# Extract individual data point

In [ ]:
data_paths = project_root / 'data' / 'OXT_All_Parameters_01282026.xlsx'
print(f"Looking for file at: {data_paths}")
data_point = pd.read_excel(data_paths)

In [ ]:
data_point

In [ ]:
# Extract the mean and SD from the data_point dataframe
data_points = data_point.copy()

# Helper function to split mean and SD
def extract_value(val):
    if pd.isna(val) or val == '-' or val == 'NaN':
        return None
    try:
        parts = str(val).split(' ± ')
        return float(parts[0])
    except:
        return None

def extract_sd(val):
    if pd.isna(val) or val == '-' or val == 'NaN':
        return None
    try:
        parts = str(val).split(' ± ')
        return float(parts[1]) if len(parts) > 1 else None
    except:
        return None

# Extract mean and SD for association rate
data_points['association_rate_mean'] = data_points['association rate'].apply(extract_value)
data_points['association_rate_SD'] = data_points['association rate'].apply(extract_sd)

# Extract mean and SD for dissociation rate
data_points['dissociation_rate_mean'] = data_points['dissociation rate'].apply(extract_value)
data_points['dissociation_rate_SD'] = data_points['dissociation rate'].apply(extract_sd)

# Extract mean and SD for Kd
data_points['Kd_mean'] = data_points['Kd'].apply(extract_value)
data_points['Kd_SD'] = data_points['Kd'].apply(extract_sd)

# Rename Symbol to parameter and select relevant columns
data_points = data_points.rename(columns={'Symbol': 'parameter'})
data_points = data_points[['parameter', 'Receptor', 'Binding Partner',
                           'association_rate_mean', 'association_rate_SD',
                           'dissociation_rate_mean', 'dissociation_rate_SD',
                           'Kd_mean', 'Kd_SD']]

# Restructure data to have each parameter as separate rows
# Define parameter mappings (maps parameter name to which columns to use)
param_mappings = {
    # Association rate parameters
    'Kong': ('association_rate_mean', 'association_rate_SD'),
    'Kact': ('association_rate_mean', 'association_rate_SD'),
    'kp': ('association_rate_mean', 'association_rate_SD'),
    'Kba': ('association_rate_mean', 'association_rate_SD'),
    # Dissociation rate parameters
    'Koffg': ('dissociation_rate_mean', 'dissociation_rate_SD'),
    'Kinact': ('dissociation_rate_mean', 'dissociation_rate_SD'),
    'kdp': ('dissociation_rate_mean', 'dissociation_rate_SD'),
    # Other kinetic parameters (using association_rate column in original data)
    'Kint': ('association_rate_mean', 'association_rate_SD'),
    'Kcons_int': ('association_rate_mean', 'association_rate_SD'),
    'Krec': ('association_rate_mean', 'association_rate_SD'),
    'Kexpt': ('association_rate_mean', 'association_rate_SD'),
    'Krgt': ('association_rate_mean', 'association_rate_SD'),
    'Kdeg': ('association_rate_mean', 'association_rate_SD'),
}

# Parameters to exclude
exclude_params = {'Koff', 'Kon'}

# Create a list to store restructured rows
restructured_rows = []

for idx, row in data_points.iterrows():
    # Parse the parameter column (could contain multiple parameters separated by comma)
    params_str = str(row['parameter']).strip()
    
    # Split by comma if multiple parameters
    if ',' in params_str:
        params = [p.strip() for p in params_str.split(',')]
    else:
        params = [params_str]
    
    # For each parameter, create a row
    for param in params:
        param_clean = param.strip()
        
        # Skip excluded parameters
        if param_clean in exclude_params:
            continue
        
        # Check if parameter is in our mappings
        if param_clean in param_mappings:
            mean_col, sd_col = param_mappings[param_clean]
            
            # Get mean and SD values
            mean_val = row[mean_col]
            sd_val = row[sd_col]
            
            # Only add if mean value is not null
            if pd.notna(mean_val):
                restructured_rows.append({
                    'Parameter': param_clean,
                    'Receptor': row['Receptor'],
                    'Binding Partner': row['Binding Partner'],
                    'Mean': mean_val,
                    'SD': sd_val
                })

# Create new dataframe and sort by Parameter to group them consistently
data_points_restructured = pd.DataFrame(restructured_rows)
data_points_restructured = data_points_restructured.sort_values('Parameter').reset_index(drop=True)
data_points_restructured

In [ ]:
# Restructure data to have each parameter as separate rows
# Define parameter mappings (maps parameter name to which columns to use)
param_mappings = {
    # Association rate parameters
    'Kong': ('association_rate_mean', 'association_rate_SD'),
    'Kact': ('association_rate_mean', 'association_rate_SD'),
    'kp': ('association_rate_mean', 'association_rate_SD'),
    'Kba': ('association_rate_mean', 'association_rate_SD'),
    # Dissociation rate parameters
    'Koffg': ('dissociation_rate_mean', 'dissociation_rate_SD'),
    'Kinact': ('dissociation_rate_mean', 'dissociation_rate_SD'),
    'kdp': ('dissociation_rate_mean', 'dissociation_rate_SD'),
    # Other kinetic parameters (using association_rate column in original data)
    'Kint': ('association_rate_mean', 'association_rate_SD'),
    'Kcons_int': ('association_rate_mean', 'association_rate_SD'),
    'Krec': ('association_rate_mean', 'association_rate_SD'),
    'Kexpt': ('association_rate_mean', 'association_rate_SD'),
    'Krgt': ('association_rate_mean', 'association_rate_SD'),
    'Kdeg': ('association_rate_mean', 'association_rate_SD'),
}

# Parameters to exclude
exclude_params = {'Koff', 'Kon'}

# Create a list to store restructured rows
restructured_rows = []

for idx, row in data_points.iterrows():
    # Parse the parameter column (could contain multiple parameters separated by comma)
    params_str = str(row['parameter']).strip()
    
    # Split by comma if multiple parameters
    if ',' in params_str:
        params = [p.strip() for p in params_str.split(',')]
    else:
        params = [params_str]
    
    # For each parameter, create a row
    for param in params:
        param_clean = param.strip()
        
        # Skip excluded parameters
        if param_clean in exclude_params:
            continue
        
        # Check if parameter is in our mappings
        if param_clean in param_mappings:
            mean_col, sd_col = param_mappings[param_clean]
            
            # Get mean and SD values
            mean_val = row[mean_col]
            sd_val = row[sd_col]
            
            # Only add if mean value is not null
            if pd.notna(mean_val):
                restructured_rows.append({
                    'Parameter': param_clean,
                    'Receptor': row['Receptor'],
                    'Binding Partner': row['Binding Partner'],
                    'Mean': mean_val,
                    'SD': sd_val
                })

# Create new dataframe and sort by Parameter to group them consistently
data_points_restructured = pd.DataFrame(restructured_rows)
data_points_restructured = data_points_restructured.sort_values('Parameter').reset_index(drop=True)
data_points_restructured

### first order

In [ ]:
# Load BHM/REM data and create comprehensive comparison plot with data points
data_model_compare = pd.read_excel(data_path, sheet_name='GPCR_parameters_beta12_geomean')

# Define first-order parameters
first_order_params = [
    'koffG', 'kact', 'kinact', 'kdp', 'kcons_int',
    'kint', 'krec', 'kexpt', 'krgt', 'kdeg'
]

# Filter to ONLY first-order parameters
data_model_compare = data_model_compare[data_model_compare['Parameter'].isin(first_order_params)].copy()

# Parameter mapping with proper LaTeX formatting
parameter_subscripts = {
    "koffG": r"$k_{\mathrm{off,G}*}$",
    "kact": r"$k_{\mathrm{act}}$",
    "kinact": r"$k_{\mathrm{inact}}$",
    "kdp": r"$k_{\mathrm{dp}}$",
    "kcons_int": r"$k_{\mathrm{cons,int}}$",
    "kint": r"$k_{\mathrm{int}}$",
    "krec": r"$k_{\mathrm{rec}}$",
    "kexpt": r"$k_{\mathrm{expt}}$",
    "krgt": r"$k_{\mathrm{rgt}}$",
    "kdeg": r"$k_{\mathrm{deg}}$"
}

# Order parameters
data_model_compare['Parameter'] = pd.Categorical(
    data_model_compare['Parameter'], 
    categories=first_order_params, 
    ordered=True
)
data_model_compare = data_model_compare.sort_values('Parameter')

# Create figure
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
plt.style.use('seaborn-v0_8-whitegrid')

# Color scheme
bhm_color = '#2E86AB'  # Blue
rem_color = '#A23B72'  # Magenta/Red
grid_color = '#E0E0E0'
data_point_color = '#656565'  # Dark gray

# Set positions
n_params = len(data_model_compare)
x_pos = np.arange(n_params)
width = 0.35

# Plot BHM estimates
bhm_means = data_model_compare['BHM (mean)']
bhm_lower = data_model_compare['BHM (lower [95% CI])']
bhm_upper = data_model_compare['BHM (upper [95% CI])']
bhm_error = [bhm_means - bhm_lower, bhm_upper - bhm_means]

bhm_bars = ax.errorbar(
    x_pos - width/2, bhm_means, 
    yerr=bhm_error,
    fmt='D',
    color=bhm_color,
    ecolor=bhm_color,
    elinewidth=1.5,
    capsize=5,
    capthick=1.5,
    markersize=9,
    markeredgewidth=1.5,
    markeredgecolor='white',
    label='BHM (95% CI)',
    alpha=0.9,
    zorder=3
)

# Plot REM estimates
rem_means = data_model_compare['REM (mean)']
rem_lower = data_model_compare['REM (lower [95% CI])']
rem_upper = data_model_compare['REM (upper [95% CI])']
rem_error = [rem_means - rem_lower, rem_upper - rem_means]

rem_bars = ax.errorbar(
    x_pos + width/2, rem_means, 
    yerr=rem_error,
    fmt='s',
    color=rem_color,
    ecolor=rem_color,
    elinewidth=1.5,
    capsize=5,
    capthick=1.5,
    markersize=8,
    markeredgewidth=1.5,
    markeredgecolor='white',
    label='REM (95% CI)',
    alpha=0.9,
    zorder=3
)

# Overlay individual data points
for param_idx, param_name in enumerate(data_model_compare['Parameter'].values):
    # Find matching data points (case-insensitive)
    param_name_lower = param_name.lower()
    matching_data = data_points_restructured[
        data_points_restructured['Parameter'].str.lower() == param_name_lower
    ]
    
    if len(matching_data) > 0:
        # Plot data points with jitter
        x_jitter = np.random.normal(param_idx, 0.04, size=len(matching_data))
        ax.scatter(x_jitter, matching_data['Mean'], 
                  alpha=0.45, s=40, color=data_point_color, 
                  edgecolors='none', zorder=2, label='Data points' if param_idx == 0 else '')

# Set log scale
ax.set_yscale('log')

# Customize axes
ax.set_ylabel('Rate Constant (min$^{-1}$) Log Scale', 
              weight='semibold', fontsize=18, labelpad=10)
ax.set_xlabel('First-Order Kinetic Parameters', 
              weight='semibold', fontsize=18, labelpad=10)
ax.set_xticks(x_pos)
ax.set_xticklabels([parameter_subscripts.get(p, p) 
                   for p in data_model_compare['Parameter']], 
                  fontsize=18, rotation=45, ha='right')

# Set axis limits
ax.set_xlim(-0.7, n_params - 0.3)
y_min = min(bhm_lower.min(), rem_lower.min()) * 0.5
y_max = max(bhm_upper.max(), rem_upper.max()) * 2.5
ax.set_ylim(y_min, y_max)

# Grid
ax.grid(True, alpha=0.4, linestyle='-', linewidth=0.7, 
        color=grid_color, which='both', axis='y')
ax.grid(True, alpha=0.2, linestyle='-', linewidth=0.5, 
        color=grid_color, which='major', axis='x')

# Legend
legend_elements = [
    Line2D([0], [0], marker='D', color='w', 
           markerfacecolor=bhm_color, markeredgecolor='white',
           markersize=10, markeredgewidth=1.5, label='BHM (95% CI)'),
    Line2D([0], [0], marker='s', color='w', 
           markerfacecolor=rem_color, markeredgecolor='white',
           markersize=9, markeredgewidth=1.5, label='REM (95% CI)'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=data_point_color, markeredgecolor='none',
           markersize=6, label='Individual Data Points')
]

ax.legend(handles=legend_elements, loc='upper right', 
          fontsize=18, framealpha=0.95, edgecolor='gray',
          fancybox=True, shadow=True)

# Adjust spines
for spine in ax.spines.values():
    spine.set_linewidth(1.2)
    spine.set_color('gray')
    spine.set_alpha(0.8)

# Add note
plt.figtext(0.5, 0.01, 
            'Data points overlaid in logarithmic scale | Error bars represent 95% confidence intervals (REM) | 95% credible intervals (BHM)',
            ha='center', fontsize=10, style='italic', alpha=0.7)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

plt.savefig('../../results/figures/python/BHM vs REM Estimates for First-Order Kinetic Rate Parameters.png',
            dpi=300, bbox_inches='tight')

# Print summary statistics
print("="*70)
print("FIRST-ORDER PARAMETERS: BHM vs REM COMPARISON WITH DATA POINTS")
print("="*70)
print(f"Number of first-order parameters: {n_params}")
print(f"\nParameter order: {data_model_compare['Parameter'].tolist()}")
print("\nBHM Statistics:")
print(f"  Range: {bhm_means.min():.2e} to {bhm_means.max():.2e}")
print(f"  Geometric mean: {np.exp(np.mean(np.log(bhm_means))):.2e}")
print("\nREM Statistics:")
print(f"  Range: {rem_means.min():.2e} to {rem_means.max():.2e}")
print(f"  Geometric mean: {np.exp(np.mean(np.log(rem_means))):.2e}")
print(f"\nIndividual Data Points Summary:")
print(f"  Total data points: {len(data_points_restructured)}")
print(f"  Parameters represented: {data_points_restructured['Parameter'].nunique()}")
print("="*70)

### second order

In [ ]:
# Create dedicated dataframe for second-order parameters (kba1, kba2, kp, kong)
# Extract data for kba 1, kba 2, kP, and konG from data_points_restructured

kba1_2_data = pd.DataFrame()

# Filter for kba with beta-arrestin binding partners
kba_data = data_points_restructured[
    data_points_restructured['Parameter'].str.lower() == 'kba'
].copy()

# Separate kba1 and kba2 based on binding partner
if len(kba_data) > 0:
    kba1_data = kba_data[
        kba_data['Binding Partner'].str.contains('beta-arrestin 1|β-arrestin 1|arrestin 1', case=False, na=False)
    ].copy()
    kba1_data['Parameter'] = 'kba 1'
    
    kba2_data = kba_data[
        kba_data['Binding Partner'].str.contains('beta-arrestin 2|β-arrestin 2|arrestin 2', case=False, na=False)
    ].copy()
    kba2_data['Parameter'] = 'kba 2'
    
    kba1_2_data = pd.concat([kba1_data, kba2_data], ignore_index=True)

# Add kp data - use case-insensitive search
kp_data = data_points_restructured[
    data_points_restructured['Parameter'].str.lower() == 'kp'
].copy()

if len(kp_data) > 0:
    kp_data['Parameter'] = 'kP'  # Match the case used in data_model_compare_2nd

# Add kong data - use case-insensitive search
kong_data = data_points_restructured[
    data_points_restructured['Parameter'].str.lower() == 'kong'
].copy()

if len(kong_data) > 0:
    kong_data['Parameter'] = 'konG'  # Match the case used in data_model_compare_2nd

# Combine all second-order data
kba1_2_data = pd.concat([kba1_2_data, kp_data, kong_data], ignore_index=True)

# Sort by parameter for consistency
kba1_2_data = kba1_2_data.sort_values('Parameter').reset_index(drop=True)

print("Second-Order Parameters Data Summary:")
print("="*70)
print(f"Total data points: {len(kba1_2_data)}")
print(f"\nBreakdown by parameter:")
for param in ['kba 1', 'kba 2', 'kP', 'konG']:
    count = len(kba1_2_data[kba1_2_data['Parameter'] == param])
    print(f"  {param}: {count} data points")

print(f"\nDataframe shape: {kba1_2_data.shape}")
print(f"\nColumns: {kba1_2_data.columns.tolist()}")
print("\nParams in data:", kba1_2_data['Parameter'].unique().tolist())
print("\nFirst few rows:")
print(kba1_2_data.head(10))


In [ ]:
# Multiply mean values by 60 for kba1 and kba2
kba_mask = kba1_2_data['Parameter'].isin(['kba 1', 'kba 2'])

# Multiply Mean by 60 for kba1 and kba2
if 'Mean' in kba1_2_data.columns:
    kba1_2_data.loc[kba_mask, 'Mean'] = kba1_2_data.loc[kba_mask, 'Mean'] * 60

# Also multiply SD (standard deviation) by 60 for consistency
if 'SD' in kba1_2_data.columns:
    kba1_2_data.loc[kba_mask, 'SD'] = kba1_2_data.loc[kba_mask, 'SD'] * 60

print("Mean and SD values multiplied by 60 for kba 1 and kba 2")
print("\nUpdated kba1_2_data:")
print(kba1_2_data)

In [ ]:
# Load BHM/REM data and create comprehensive comparison plot for SECOND-ORDER parameters with broken axis
data_model_compare_2nd = pd.read_excel(data_path, sheet_name='GPCR_parameters_beta12_geomean')

# Define second-order parameters (parameters that have concentration dependence or include bimolecular elements)
second_order_params = [
    'konG', 'kP', 'kba 1', 'kba 2',    # Bimolecular association and binding parameters
]

# Filter to ONLY second-order parameters that exist in the data
data_model_compare_2nd = data_model_compare_2nd[data_model_compare_2nd['Parameter'].isin(second_order_params)].copy()

if len(data_model_compare_2nd) == 0:
    print("No classic second-order parameters (kba, kP) found.")
    print("Using available first-order kinetic components for comparison:")
    # Alternative: use other kinetic parameters for display
    alt_params = ['kint', 'krec', 'kexpt', 'krgt', 'kdeg']
    data_model_compare_2nd = pd.read_excel(data_path, sheet_name='GPCR_parameters_beta12_geomean')
    data_model_compare_2nd = data_model_compare_2nd[data_model_compare_2nd['Parameter'].isin(alt_params)].copy()
    if len(data_model_compare_2nd) == 0:
        print("Unable to create second-order plot - parameters not found in spreadsheet")
    else:
        second_order_params = alt_params

if len(data_model_compare_2nd) > 0:
    # Parameter mapping with proper LaTeX formatting
    parameter_subscripts_2nd = {
        "kP": r"$k_P$",
        "konG": r"$k_{\mathrm{on,G}*}$",
        "kba 1": r"$k_{\mathrm{ba,1}}$",
        "kba 2": r"$k_{\mathrm{ba,2}}$",
        "kint": r"$k_{\mathrm{int}}$",
        "krec": r"$k_{\mathrm{rec}}$",
        "kexpt": r"$k_{\mathrm{expt}}$",
        "krgt": r"$k_{\mathrm{rgt}}$",
        "kdeg": r"$k_{\mathrm{deg}}$"
    }
    
    # Order parameters
    existing_params = [p for p in second_order_params if p in data_model_compare_2nd['Parameter'].values]
    data_model_compare_2nd['Parameter'] = pd.Categorical(
        data_model_compare_2nd['Parameter'], 
        categories=existing_params, 
        ordered=True
    )
    data_model_compare_2nd = data_model_compare_2nd.sort_values('Parameter')
    
    # Create figure with broken axis (two subplots)
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(14, 8))
    fig.subplots_adjust(hspace=0.05)  # Adjust space between axes
    
    # Color scheme
    bhm_color = '#2E86AB'  # Blue
    rem_color = '#A23B72'  # Magenta/Red
    grid_color = '#E0E0E0'
    
    # Set positions
    n_params = len(data_model_compare_2nd)
    x_pos = np.arange(n_params)
    width = 0.45  # Slightly wider for better separation
    
    # Helper function to plot estimates on given axis
    def plot_estimates(ax):
        # Plot BHM estimates
        bhm_means = data_model_compare_2nd['BHM (mean)']
        bhm_lower = data_model_compare_2nd['BHM (lower [95% CI])']
        bhm_upper = data_model_compare_2nd['BHM (upper [95% CI])']
        bhm_error = [bhm_means - bhm_lower, bhm_upper - bhm_means]
        
        ax.errorbar(
            x_pos - width/2, bhm_means, 
            yerr=bhm_error,
            fmt='D',  # Diamond marker
            color=bhm_color,
            ecolor=bhm_color,
            elinewidth=1.5,
            capsize=5,
            capthick=1.5,
            markersize=9,
            markeredgewidth=1.5,
            markeredgecolor='white',
            label='BHM',
            alpha=0.9,
            zorder=3
        )
        
        # Plot REM estimates
        rem_means = data_model_compare_2nd['REM (mean)']
        rem_lower = data_model_compare_2nd['REM (lower [95% CI])']
        rem_upper = data_model_compare_2nd['REM (upper [95% CI])']
        rem_error = [rem_means - rem_lower, rem_upper - rem_means]
        
        ax.errorbar(
            x_pos + width/2, rem_means, 
            yerr=rem_error,
            fmt='s',  # Square marker
            color=rem_color,
            ecolor=rem_color,
            elinewidth=1.5,
            capsize=5,
            capthick=1.5,
            markersize=8,
            markeredgewidth=1.5,
            markeredgecolor='white',
            label='REM',
            alpha=0.9,
            zorder=3
        )
        
        # Overlay individual data points
        for param_idx, param_name in enumerate(data_model_compare_2nd['Parameter'].values):
            # Find matching data points (case-insensitive)
            param_name_lower = param_name.lower()
            matching_data = kba1_2_data[
                kba1_2_data['Parameter'].str.lower() == param_name_lower
            ]
            
            if len(matching_data) > 0:
                # Plot data points with jitter (centered on the parameter position)
                x_jitter = np.random.normal(x_pos[param_idx], 0.04, size=len(matching_data))
                ax.scatter(x_jitter, matching_data['Mean'], 
                        alpha=0.45, s=40, color=data_point_color, 
                        edgecolors='none', zorder=2, label='Data points' if param_idx == 0 else '')
        
        # Set log scale
        ax.set_yscale('log')
        
        # Improve grid
        ax.grid(True, alpha=0.4, linestyle='-', linewidth=0.7, 
                color=grid_color, which='both', axis='y')
        ax.grid(True, alpha=0.2, linestyle='-', linewidth=0.5, 
                color=grid_color, which='major', axis='x')
        
        # Add subtle horizontal lines for major ticks
        for tick in ax.get_yticks(minor=False):
            ax.axhline(y=tick, alpha=0.15, linestyle='-', 
                       linewidth=0.5, color='black', zorder=1)
    
    # Plot data on both axes
    for ax in [ax1, ax2]:
        plot_estimates(ax)
    
    # Get data range for setting appropriate breaks
    bhm_means = data_model_compare_2nd['BHM (mean)']
    rem_means = data_model_compare_2nd['REM (mean)']
    max_val = max(bhm_means.max(), rem_means.max())
    min_val = min(bhm_means.min(), rem_means.min())
    
    print(f"Data range: {min_val:.2e} to {max_val:.2e}")
    
    # Set y-limits for broken axis effect
    ax1.set_ylim(1e4, 1e8)  # Upper part (higher values)
    ax2.set_ylim(1e-1, 1e1)  # Lower part (lower values)
    
    # Hide spines between axes for broken axis effect
    ax1.spines['bottom'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax1.xaxis.tick_top()
    ax1.tick_params(labeltop=False, top=False)  # Hide top ticks
    ax2.xaxis.tick_bottom()
    
    # Add diagonal break markers
    d = 0.015  # Size of diagonal lines
    kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False, linewidth=1.5)
    ax1.plot((-d, +d), (-d, +d), **kwargs)        # Top-left diagonal
    ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)  # Top-right diagonal
    
    kwargs.update(transform=ax2.transAxes)
    ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # Bottom-left diagonal
    ax2.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)  # Bottom-right diagonal
    
    # Customize x-axis labels (only on bottom axis)
    subscript_labels = [parameter_subscripts_2nd.get(p, p) 
                       for p in data_model_compare_2nd['Parameter']]
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(subscript_labels, fontsize=18, rotation=45, ha='right')
    
    # Set axis labels
    ax2.set_xlabel('Second-Order Kinetic Parameters', 
                  weight='semibold', fontsize=18, labelpad=10)
    
    fig.text(-0.009, 0.5, 'Rate Constant (M$^{-1}$min$^{-1}$) Log Scale', 
            va='center', rotation='vertical', 
            weight='semibold', fontsize=18)
    
    # Set axis limits with padding
    ax1.set_xlim(-0.7, n_params - 0.3)
    ax2.set_xlim(-0.7, n_params - 0.3)
    
    # Adjust spines (borders) for both axes
    for ax in [ax1, ax2]:
        for spine in ax.spines.values():
            spine.set_linewidth(1.2)
            spine.set_color('gray')
            spine.set_alpha(0.8)
    
    # Add legend with custom styling (only on top axis)
    legend_elements = [
        Line2D([0], [0], marker='D', color='w', 
               markerfacecolor=bhm_color, markeredgecolor='white',
               markersize=10, markeredgewidth=1.5, label='BHM'),
        Line2D([0], [0], marker='s', color='w', 
               markerfacecolor=rem_color, markeredgecolor='white',
               markersize=9, markeredgewidth=1.5, label='REM'),
        Line2D([0], [0], marker='o', color='w', 
               markerfacecolor=data_point_color, markeredgecolor='none',
               markersize=6, alpha=0.45, label='Individual data points'),
        Patch(facecolor='none', edgecolor='none', 
              label='Error bars: 95% CI')
    ]
    
    ax1.legend(handles=legend_elements, loc='upper right', 
              fontsize=18, framealpha=0.95, edgecolor='gray',
              fancybox=True, shadow=True)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Second-order parameters plot created with broken axis and {n_params} parameters ({', '.join(existing_params)})")

    print(Path.cwd())

    plt.savefig('../../results/figures/python/BHM vs REM Estimates for Second-Order Kinetic Rate Parameters.png',
            dpi=300, bbox_inches='tight')

# Refinded Analysis seperating by units

## REM vs BHM graphs 1st order

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Load and prepare data
data_2 = pd.read_excel(data_path, sheet_name='GPCR_parameters_beta12_geomean')

# Define first-order parameters
first_order_params = [
    'koffG', 'kact', 'kinact', 'kdp', 'kcons_int',
    'kint', 'krec', 'kexpt', 'krgt', 'kdeg'
]

# Filter to include ONLY first-order parameters
data_2 = data_2[data_2['Parameter'].isin(first_order_params)]

# Parameter mapping with proper LaTeX formatting
parameter_subscripts = {
    "koffG": r"$k_{\mathrm{off,G}}$",
    "kact": r"$k_{\mathrm{act}}$",
    "kinact": r"$k_{\mathrm{inact}}$",
    "kdp": r"$k_{\mathrm{dp}}$",
    "kcons_int": r"$k_{\mathrm{cons,int}}$",
    "kint": r"$k_{\mathrm{int}}$",
    "krec": r"$k_{\mathrm{rec}}$",
    "kexpt": r"$k_{\mathrm{expt}}$",
    "krgt": r"$k_{\mathrm{rgt}}$",
    "kdeg": r"$k_{\mathrm{deg}}$"
}

# Order parameters as in the list (ensuring consistent order)
data_2['Parameter'] = pd.Categorical(
    data_2['Parameter'], 
    categories=first_order_params, 
    ordered=True
)
data_2 = data_2.sort_values('Parameter')

# Create figure with better proportions
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
plt.style.use('seaborn-v0_8-whitegrid')  # Clean, modern style
#plt.style.use('ggplot')  # R-style
#plt.style.use('bmh')  # Bayesian Methods for Hackers style

# Color scheme (colorblind friendly)
bhm_color = '#2E86AB'  # Blue
rem_color = '#A23B72'  # Magenta/Red
grid_color = '#E0E0E0'

# Set positions
n_params = len(data_2)
x_pos = np.arange(n_params)
width = 0.45  # Slightly wider for better separation

# Plot BHM estimates
bhm_means = data_2['BHM (mean)']
bhm_lower = data_2['BHM (lower [95% CI])']
bhm_upper = data_2['BHM (upper [95% CI])']
bhm_error = [bhm_means - bhm_lower, bhm_upper - bhm_means]

bhm_bars = ax.errorbar(
    x_pos - width/2, bhm_means, 
    yerr=bhm_error,
    fmt='D',  # Diamond marker
    color=bhm_color,
    ecolor=bhm_color,
    elinewidth=1.5,
    capsize=5,
    capthick=1.5,
    markersize=9,
    markeredgewidth=1.5,
    markeredgecolor='white',
    label='BHM',
    alpha=0.9,
    zorder=3
)

# Plot REM estimates
rem_means = data_2['REM (mean)']
rem_lower = data_2['REM (lower [95% CI])']
rem_upper = data_2['REM (upper [95% CI])']
rem_error = [rem_means - rem_lower, rem_upper - rem_means]

rem_bars = ax.errorbar(
    x_pos + width/2, rem_means, 
    yerr=rem_error,
    fmt='s',  # Square marker
    color=rem_color,
    ecolor=rem_color,
    elinewidth=1.5,
    capsize=5,
    capthick=1.5,
    markersize=8,
    markeredgewidth=1.5,
    markeredgecolor='white',
    label='REM',
    alpha=0.9,
    zorder=3
)

# Set log scale with better formatting
ax.set_yscale('log')
#ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f'{y:.1e}'))

# Customize y-axis
ax.set_ylabel('Rate Constant (min$^{-1}$) Log Scale', 
              weight='semibold', fontsize=13, labelpad=10)

# Customize x-axis
ax.set_xlabel('First-Order Kinetic Parameters', 
              weight='semibold', fontsize=13, labelpad=10)
ax.set_xticks(x_pos)
ax.set_xticklabels([parameter_subscripts.get(p, p) 
                   for p in data_2['Parameter']], 
                  fontsize=12, rotation=45, ha='right')

# Set axis limits with padding
ax.set_xlim(-0.7, n_params - 0.3)
y_min = min(bhm_lower.min(), rem_lower.min()) * 0.5
y_max = max(bhm_upper.max(), rem_upper.max()) * 2
ax.set_ylim(y_min, y_max)

# Improve grid
ax.grid(True, alpha=0.4, linestyle='-', linewidth=0.7, 
        color=grid_color, which='both', axis='y')
ax.grid(True, alpha=0.2, linestyle='-', linewidth=0.5, 
        color=grid_color, which='major', axis='x')

# Add legend with custom styling
legend_elements = [
    Line2D([0], [0], marker='D', color='w', 
           markerfacecolor=bhm_color, markeredgecolor='white',
           markersize=10, markeredgewidth=1.5, label='BHM'),
    Line2D([0], [0], marker='s', color='w', 
           markerfacecolor=rem_color, markeredgecolor='white',
           markersize=9, markeredgewidth=1.5, label='REM'),
    Patch(facecolor='none', edgecolor='none', 
          label='Error bars: 95% CI')
]

ax.legend(handles=legend_elements, loc='upper right', 
          fontsize=11, framealpha=0.95, edgecolor='gray',
          fancybox=True, shadow=True)

# Add subtle horizontal lines for major ticks
for tick in ax.get_yticks(minor=False):
    ax.axhline(y=tick, alpha=0.15, linestyle='-', 
               linewidth=0.5, color='black', zorder=1)

# Add title
#plt.title('Comparison of BHM and REM Estimates for First-Order Kinetic Parameters', fontsize=14, weight='bold', pad=15)

# Add a subtle note about confidence intervals
plt.figtext(0.5, 0.01, 
            'Error bars represent 95% confidence intervals REM | 95% credible intervals BHM',
            ha='center', fontsize=9, style='italic', alpha=0.7)

# Adjust spines (borders)
for spine in ax.spines.values():
    spine.set_linewidth(1.2)
    spine.set_color('gray')
    spine.set_alpha(0.8)

# Tight layout with extra padding
plt.tight_layout(rect=[0, 0.03, 1, 0.97])

# Alternative: Use plt.subplots_adjust for more control
# plt.subplots_adjust(left=0.08, right=0.95, top=0.92, bottom=0.15)

plt.show()

# Print enhanced summary statistics
print("="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"Number of first-order parameters: {n_params}")
print(f"\nParameter order: {data_2['Parameter'].tolist()}")
print("\nBHM Statistics:")
print(f"  Range: {bhm_means.min():.2e} to {bhm_means.max():.2e}")
print(f"  Geometric mean: {np.exp(np.mean(np.log(bhm_means))):.2e}")
print("\nREM Statistics:")
print(f"  Range: {rem_means.min():.2e} to {rem_means.max():.2e}")
print(f"  Geometric mean: {np.exp(np.mean(np.log(rem_means))):.2e}")
print("="*60)

In [ ]:
comparative_results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Load and prepare data
data_2 = pd.read_excel(data_path, sheet_name='GPCR_parameters_beta12_geomean')

# Define first-order parameters
first_order_params = [
    'koffG', 'kact', 'kinact', 'kdp', 'kcons_int',
    'kint', 'krec', 'kexpt', 'krgt', 'kdeg'
]

# Filter to include ONLY first-order parameters
data_2 = data_2[data_2['Parameter'].isin(first_order_params)]

# Filter for first-order parameters in comparative_results
comparative_results_filtered = comparative_results_all[comparative_results_all['Parameter'].isin(first_order_params)]

# Merge the Z-score and p-value information
data_2 = pd.merge(data_2, comparative_results_filtered[['Parameter', 'Z_score', 'P_value', 'Significant']], 
                  on='Parameter', how='left')

# Parameter mapping with proper LaTeX formatting
parameter_subscripts = {
    "koffG": r"$k_{\mathrm{off,G}}$",
    "kact": r"$k_{\mathrm{act}}$",
    "kinact": r"$k_{\mathrm{inact}}$",
    "kdp": r"$k_{\mathrm{dp}}$",
    "kcons_int": r"$k_{\mathrm{cons,int}}$",
    "kint": r"$k_{\mathrm{int}}$",
    "krec": r"$k_{\mathrm{rec}}$",
    "kexpt": r"$k_{\mathrm{expt}}$",
    "krgt": r"$k_{\mathrm{rgt}}$",
    "kdeg": r"$k_{\mathrm{deg}}$"
}

# Order parameters as in the list (ensuring consistent order)
data_2['Parameter'] = pd.Categorical(
    data_2['Parameter'], 
    categories=first_order_params, 
    ordered=True
)
data_2 = data_2.sort_values('Parameter')

# Create figure with better proportions
fig, ax = plt.subplots(1, 1, figsize=(16, 10))  # Increased height for annotations
plt.style.use('seaborn-v0_8-whitegrid')

# Color scheme (colorblind friendly)
bhm_color = '#2E86AB'  # Blue
rem_color = '#A23B72'  # Magenta/Red
significant_color = "#41423D"  # Orange for significant markers
grid_color = '#E0E0E0'

# Set positions
n_params = len(data_2)
x_pos = np.arange(n_params)
width = 0.45  # Slightly wider for better separation

# Plot BHM estimates
bhm_means = data_2['BHM (mean)']
bhm_lower = data_2['BHM (lower [95% CI])']
bhm_upper = data_2['BHM (upper [95% CI])']
bhm_error = [bhm_means - bhm_lower, bhm_upper - bhm_means]

bhm_bars = ax.errorbar(
    x_pos - width/2, bhm_means, 
    yerr=bhm_error,
    fmt='D',  # Diamond marker
    color=bhm_color,
    ecolor=bhm_color,
    elinewidth=1.5,
    capsize=5,
    capthick=1.5,
    markersize=9,
    markeredgewidth=1.5,
    markeredgecolor='white',
    label='BHM',
    alpha=0.9,
    zorder=3
)

# Plot REM estimates
rem_means = data_2['REM (mean)']
rem_lower = data_2['REM (lower [95% CI])']
rem_upper = data_2['REM (upper [95% CI])']
rem_error = [rem_means - rem_lower, rem_upper - rem_means]

rem_bars = ax.errorbar(
    x_pos + width/2, rem_means, 
    yerr=rem_error,
    fmt='s',  # Square marker
    color=rem_color,
    ecolor=rem_color,
    elinewidth=1.5,
    capsize=5,
    capthick=1.5,
    markersize=8,
    markeredgewidth=1.5,
    markeredgecolor='white',
    label='REM',
    alpha=0.9,
    zorder=3
)

# Set log scale with better formatting
ax.set_yscale('log')

# Customize y-axis
ax.set_ylabel('Rate Constant (s$^{-1}$) Log Scale', 
              weight='semibold', fontsize=13, labelpad=10)

# Customize x-axis
ax.set_xlabel('First-Order Kinetic Parameters', 
              weight='semibold', fontsize=13, labelpad=10)
ax.set_xticks(x_pos)
ax.set_xticklabels([parameter_subscripts.get(p, p) 
                   for p in data_2['Parameter']], 
                  fontsize=12, rotation=45, ha='right')

# Set axis limits with padding
ax.set_xlim(-0.7, n_params - 0.3)
y_min = min(bhm_lower.min(), rem_lower.min()) * 0.5
y_max = max(bhm_upper.max(), rem_upper.max()) * 2
ax.set_ylim(y_min, y_max)

# Add annotations for significant differences
for i, (param, significant, z_score, p_value) in enumerate(zip(
    data_2['Parameter'], 
    data_2['Significant'], 
    data_2['Z_score'], 
    data_2['P_value']
)):
    if significant and not pd.isna(significant):
        # Determine the highest point between BHM and REM for annotation placement
        bhm_val = bhm_means.iloc[i]
        rem_val = rem_means.iloc[i]
        max_val = max(bhm_val, rem_val)
        
        # Add annotation with Z-score and p-value
        annotation_text = f"Z = {z_score:.2f}\n p < 0.05"
        
        # Adjust vertical position based on log scale
        y_offset = max_val * 2.1  # Multiplicative offset for log scale       
        
        # Add text box with Z-score and p-value
        ax.annotate(annotation_text,
                    xy=(x_pos[i], y_offset),
                    xytext=(0, 5),
                    textcoords='offset points',
                    ha='center',
                    va='bottom',
                    fontsize=9,
                    bbox=dict(boxstyle="round,pad= .5", 
                              facecolor='white', 
                              edgecolor=significant_color,
                              alpha=0.9,
                              linewidth=1.5),
                    zorder=4)
        
        # Add a subtle line connecting the annotation to the highest bar
        ax.plot([x_pos[i], x_pos[i]], [max_val * 1.05, y_offset * 0.95], 
                color=significant_color, alpha=0.5, linewidth=1, linestyle='-')

# Improve grid
ax.grid(True, alpha=0.4, linestyle='-', linewidth=0.7, 
        color=grid_color, which='both', axis='y')
ax.grid(True, alpha=0.2, linestyle='-', linewidth=0.5, 
        color=grid_color, which='major', axis='x')

# Add legend with custom styling
legend_elements = [
    Line2D([0], [0], marker='D', color='w', 
           markerfacecolor=bhm_color, markeredgecolor='white',
           markersize=10, markeredgewidth=1.5, label='BHM'),
    Line2D([0], [0], marker='s', color='w', 
           markerfacecolor=rem_color, markeredgecolor='white',
           markersize=9, markeredgewidth=1.5, label='REM'),
]

ax.legend(handles=legend_elements, loc='upper right', 
          fontsize=11, framealpha=0.95, edgecolor='gray',
          fancybox=True, shadow=True, ncol=2)

# Add subtle horizontal lines for major ticks
for tick in ax.get_yticks(minor=False):
    ax.axhline(y=tick, alpha=0.15, linestyle='-', 
               linewidth=0.5, color='black', zorder=1)

# Add a subtle note about significance testing
plt.figtext(0.5, 0.01, 
            'Error bars: 95% credible intervals (BHM) | 95% confidence intervals (REM)\n'
            '* indicates significant difference between BHM and REM estimates (p < 0.05)',
            ha='center', fontsize=9, style='italic', alpha=0.8)

# Adjust spines (borders)
for spine in ax.spines.values():
    spine.set_linewidth(1.2)
    spine.set_color('gray')
    spine.set_alpha(0.8)

# Tight layout with extra padding
plt.tight_layout(rect=[0, 0.03, 1, 0.97])

plt.show()

# Print enhanced summary statistics with significance information
print("="*80)
print("FIRST-ORDER PARAMETERS WITH SIGNIFICANCE TESTING")
print("="*80)
print(f"Number of first-order parameters: {n_params}")
print(f"Number of significant differences: {data_2['Significant'].sum()}")

print("\nDetailed Statistics:")
for i, row in data_2.iterrows():
    param = row['Parameter']
    bhm_mean = row['BHM (mean)']
    rem_mean = row['REM (mean)']
    z_score = row['Z_score']
    p_value = row['P_value']
    significant = row['Significant']
    
    if not pd.isna(z_score):
        sig_marker = "**" if significant else ""
        print(f"\n{parameter_subscripts.get(param, param)} {sig_marker}")
        print(f"  BHM: {bhm_mean:.2e} | REM: {rem_mean:.2e}")
        print(f"  Z-score: {z_score:.3f} | p-value: {p_value:.3e}")
        if significant:
            ratio = max(bhm_mean, rem_mean) / min(bhm_mean, rem_mean)
            print(f"  Ratio (max/min): {ratio:.2f}x")
    else:
        print(f"\n{parameter_subscripts.get(param, param)}: No comparison available")

print("\n" + "="*80)

## REM vs BHM graphs 2nd order

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Bring in the data
data_2 = pd.read_excel(data_path, sheet_name='GPCR_parameters_beta12_geomean')

# Define second-order parameters
second_order_params = ['konG', 'kP', 'kba 1', 'kba 2']

# Filter for second-order parameters only
data_2 = data_2[data_2['Parameter'].isin(second_order_params)]

# Parameter mapping with proper LaTeX formatting
parameter_subscripts = {
    "konG": r"$k_{\mathrm{on,G}}$",
    "kP": r"$k_{\mathrm{P}}$",
    "kba 1": r"$k_{\mathrm{ba,1}}$",
    "kba 2": r"$k_{\mathrm{ba,2}}$"
}

# Ensure consistent order
data_2['Parameter'] = pd.Categorical(
    data_2['Parameter'], 
    categories=second_order_params, 
    ordered=True
)
data_2 = data_2.sort_values('Parameter')

# Create figure with two subplots for broken axis
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(14, 10))
fig.subplots_adjust(hspace=0.05)  # Adjust space between axes

# Color scheme (colorblind friendly - consistent with first plot)
bhm_color = '#2E86AB'  # Blue
rem_color = '#A23B72'  # Magenta/Red
grid_color = '#E0E0E0'

# Set positions for the groups
n_params = len(data_2)
x_pos = np.arange(n_params)
width = 0.45  # Slightly wider for better separation

# Helper function to plot data (for both axes)
def plot_estimates(ax):
    # Plot BHM estimates
    bhm_means = data_2['BHM (mean)']
    bhm_lower = data_2['BHM (lower [95% CI])']
    bhm_upper = data_2['BHM (upper [95% CI])']
    bhm_error = [bhm_means - bhm_lower, bhm_upper - bhm_means]
    
    ax.errorbar(
        x_pos - width/2, bhm_means, 
        yerr=bhm_error,
        fmt='D',  # Diamond marker
        color=bhm_color,
        ecolor=bhm_color,
        elinewidth=1.5,
        capsize=5,
        capthick=1.5,
        markersize=9,
        markeredgewidth=1.5,
        markeredgecolor='white',
        label='BHM',
        alpha=0.9,
        zorder=3
    )
    
    # Plot REM estimates
    rem_means = data_2['REM (mean)']
    rem_lower = data_2['REM (lower [95% CI])']
    rem_upper = data_2['REM (upper [95% CI])']
    rem_error = [rem_means - rem_lower, rem_upper - rem_means]
    
    ax.errorbar(
        x_pos + width/2, rem_means, 
        yerr=rem_error,
        fmt='s',  # Square marker
        color=rem_color,
        ecolor=rem_color,
        elinewidth=1.5,
        capsize=5,
        capthick=1.5,
        markersize=8,
        markeredgewidth=1.5,
        markeredgecolor='white',
        label='REM',
        alpha=0.9,
        zorder=3
    )
    
    # Set log scale
    ax.set_yscale('log')
    
    # Add grid
    ax.grid(True, alpha=0.4, linestyle='-', linewidth=0.7, 
            color=grid_color, which='both', axis='y')
    ax.grid(True, alpha=0.2, linestyle='-', linewidth=0.5, 
            color=grid_color, which='major', axis='x')
    
    # Add subtle horizontal lines for major ticks
    for tick in ax.get_yticks(minor=False):
        ax.axhline(y=tick, alpha=0.15, linestyle='-', 
                   linewidth=0.5, color='black', zorder=1)

# Plot data on both axes
for ax in [ax1, ax2]:
    plot_estimates(ax)

# Get data range for setting appropriate breaks
bhm_means = data_2['BHM (mean)']
rem_means = data_2['REM (mean)']
max_val = max(bhm_means.max(), rem_means.max())
min_val = min(bhm_means.min(), rem_means.min())

print(f"Data range: {min_val:.2e} to {max_val:.2e}")

# Set y-limits for broken axis effect
# Adjust these values based on your actual data range
ax1.set_ylim(1e4, 1e7)  # Upper part (higher values)
ax2.set_ylim(1e-1, 1e1)  # Lower part (lower values) - adjust as needed

# Hide spines between axes for broken axis effect
ax1.spines['bottom'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax1.xaxis.tick_top()
ax1.tick_params(labeltop=False, top=False)  # Hide top ticks
ax2.xaxis.tick_bottom()

# Add diagonal break markers
d = 0.015  # Size of diagonal lines
kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False, linewidth=1.5)
ax1.plot((-d, +d), (-d, +d), **kwargs)        # Top-left diagonal
ax1.plot((1 - d, 1 + d), (-d, +d), **kwargs)  # Top-right diagonal

kwargs.update(transform=ax2.transAxes)
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # Bottom-left diagonal
ax2.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)  # Bottom-right diagonal

# Customize x-axis labels
subscript_labels = [parameter_subscripts[param] for param in data_2['Parameter']]
ax2.set_xticks(x_pos)
ax2.set_xticklabels(subscript_labels, rotation=0, ha='center', fontsize=12)

# Set x-axis label
ax2.set_xlabel('Second-Order Kinetic Parameters', 
               weight='semibold', fontsize=13, labelpad=10)

# Set y-axis labels
fig.text(0.04, 0.5, 'Rate Constant (M$^{-1}$min$^{-1}$) Log Scale', 
         va='center', rotation='vertical', 
         weight='semibold', fontsize=13)

# Adjust spines (borders) for both axes
for ax in [ax1, ax2]:
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
        spine.set_color('gray')
        spine.set_alpha(0.8)

# Create custom legend
legend_elements = [
    Line2D([0], [0], marker='D', color='w', 
           markerfacecolor=bhm_color, markeredgecolor='white',
           markersize=10, markeredgewidth=1.5, label='BHM'),
    Line2D([0], [0], marker='s', color='w', 
           markerfacecolor=rem_color, markeredgecolor='white',
           markersize=9, markeredgewidth=1.5, label='REM'),
    Patch(facecolor='none', edgecolor='none', 
          label='Error bars: 95% CI')
]

# Add legend to the figure 
ax1.legend(handles=legend_elements, loc='upper right', 
          fontsize=11, framealpha=0.95, edgecolor='gray',
          fancybox=True, shadow=True)

# Add a subtle note about confidence intervals
fig.text(0.5, 0.01, 
         'Error bars represent 95% confidence intervals REM | 95% credible intervals BHM',
         ha='center', fontsize=9, style='italic', alpha=0.7)

# Adjust subplot positions for better spacing
ax1.set_position([0.125, 0.55, 0.775, 0.35])  # [left, bottom, width, height]
ax2.set_position([0.125, 0.1, 0.775, 0.35])

plt.show()

# Print enhanced summary statistics
print("="*60)
print("SECOND-ORDER PARAMETERS SUMMARY")
print("="*60)
print(f"Number of second-order parameters: {n_params}")
print(f"\nParameter order: {data_2['Parameter'].tolist()}")
print("\nBHM Statistics:")
print(f"  Range: {bhm_means.min():.2e} to {bhm_means.max():.2e}")
print(f"  Geometric mean: {np.exp(np.mean(np.log(bhm_means))):.2e}")
print("\nREM Statistics:")
print(f"  Range: {rem_means.min():.2e} to {rem_means.max():.2e}")
print(f"  Geometric mean: {np.exp(np.mean(np.log(rem_means))):.2e}")
print("="*60)

# ligands sub analysis

## import data

In [ ]:
data_ligands= pd.read_excel(data_path, sheet_name='ligands')

## graph ligand data

#### pie and bar chart with arrows

In [ ]:
def create_dual_chart_with_arrows(data_series, bar_series=None, threshold_percent=2.3, 
                                 pie_title="Distribution", bar_title="Top 15 Items",
                                 figsize=(18, 7), show_stats=True):
    """
    Create a dual chart with pie chart (with threshold grouping) and bar chart
    Uses leader lines (arrows) for pie chart labels to avoid overlap
    """
    # Create figure
    if bar_series is not None:
        fig, axes = plt.subplots(1, 2, figsize=figsize)
    else:
        fig, axes = plt.subplots(1, 1, figsize=(figsize[0]//2, figsize[1]))
        axes = [axes]
    
    # --- PIE CHART WITH LEADER LINES ---
    counts = data_series.value_counts()
    total = counts.sum()
    
    # Group small slices
    percentages = (counts / total) * 100
    main = counts[percentages >= threshold_percent]
    other = counts[percentages < threshold_percent]
    other_total = other.sum()
    
    # Prepare pie data
    if len(other) > 0:
        pie_data = pd.concat([main, pd.Series({'Other': other_total})])
    else:
        pie_data = counts
    
    # Create labels with improved formatting
    pie_percentages = (pie_data / total) * 100
    labels = []
    for name, count in pie_data.items():
        percent = pie_percentages[name]
        if name == 'Other' and len(other) > 0:
            labels.append(f'Other\n({len(other)} items, {percent:.1f}%)')
        else:
            # Truncate long names
            display_name = name if len(name) <= 15 else name[:12] + '...'
            labels.append(f'{display_name}\n{count} ({percent:.1f}%)')
    
    # Plot pie with explode for better visibility
    colors = plt.cm.Set3(np.arange(len(pie_data)))
    explode = [0.05] * len(pie_data)
    if len(other) > 0:
        explode[-1] = 0.1
    
    # Plot pie without labels first
    wedges, texts = axes[0].pie(pie_data.values, colors=colors,
                                explode=explode, startangle=90,
                                wedgeprops=dict(edgecolor='w', linewidth=1))
    
    # Add labels with leader lines
    from matplotlib.patches import FancyArrowPatch
    
    # Create a bbox for annotation positioning
    bbox_props = dict(boxstyle="round,pad=0.3", fc="w", ec="black", lw=0.72)
    kw = dict(arrowprops=dict(arrowstyle="-"), bbox=bbox_props, zorder=0, 
              va="center", fontsize=10, weight='bold')
    
    for i, (wedge, label) in enumerate(zip(wedges, labels)):
        ang = (wedge.theta2 + wedge.theta1) / 2.
        y = np.sin(np.deg2rad(ang))
        x = np.cos(np.deg2rad(ang))
        
        # Determine annotation position based on angle
        horizontalalignment = {-1: "right", 1: "left"}[int(np.sign(x))]
        connectionstyle = f"angle,angleA=0,angleB={ang}"
        kw["arrowprops"].update({"connectionstyle": connectionstyle})
        
        # Position annotation
        annotation_distance = 1.3
        ax = axes[0]
        ax.annotate(label, xy=(x, y), xytext=(annotation_distance*x, annotation_distance*y),
                   horizontalalignment=horizontalalignment, **kw)
    
    axes[0].set_title(f'{pie_title}\n(Threshold: {threshold_percent}%)', fontsize=14)
    axes[0].set_aspect('equal')  # Make pie chart circular
    
    # --- BAR CHART (if provided) ---
    if bar_series is not None:
        # Get top 15 for bar chart
        top_15 = bar_series.value_counts().head(15)
        
        # Create bar chart
        colors_bar = plt.cm.viridis(np.linspace(0, 1, 15))
        bars = axes[1].barh(top_15.index[::-1], top_15.values[::-1],
                           color=colors_bar)
        
        # Add value labels
        max_val = top_15.max()
        for bar in bars:
            width = bar.get_width()
            if width > max_val * 0.3:
                axes[1].text(width * 0.95, bar.get_y() + bar.get_height()/2,
                            f'{int(width)}', va='center', ha='right',
                            color='black', fontsize=9, fontweight='bold')
            else:
                axes[1].text(width + (max_val * 0.02), bar.get_y() + bar.get_height()/2,
                            f'{int(width)}', va='center', ha='left',
                            color='black', fontsize=9, fontweight='bold')
        
        axes[1].set_xlabel('Count')
        axes[1].set_title(bar_title, fontsize=14)
        axes[1].grid(True, alpha=0.3, axis='x')
        axes[1].set_xlim(0, max_val * 1.18)
    
    plt.tight_layout()
    plt.show()
    
    # Statistics (if requested)
    if show_stats:
        print("📊 STATISTICS")
        print("="*50)
        print(f"Pie Chart Data ({data_series.name}):")
        print(f"  Total records: {total}")
        print(f"  Unique items: {len(counts)}")
        print(f"  Items ≥{threshold_percent}%: {len(main)}")
        print(f"  Items <{threshold_percent}%: {len(other)}")
        
        if len(main) > 0:
            print(f"\nTop items (shown individually):")
            for i, (item, count) in enumerate(main.head(10).items(), 1):
                percent = (count / total) * 100
                print(f"  {i}. {item}: {count} ({percent:.1f}%)")
        
        if bar_series is not None:
            print(f"\nBar Chart Data ({bar_series.name}):")
            print(f"  Total records: {len(bar_series)}")
            print(f"  Unique items: {bar_series.nunique()}")
            print(f"  Top 5 items:")
            for i, (item, count) in enumerate(bar_series.value_counts().head(5).items(), 1):
                print(f"  {i}. {item}: {count}")
    
    return main, other, fig

In [ ]:
# Test the smart labels version
main_receptors, other_receptors, fig = create_dual_chart_with_arrows(
    data_series=data_ligands['receptor'],
    bar_series=data_ligands['ligand'],
    threshold_percent=2.3,
    pie_title="Receptor Distribution",
    bar_title="Top 15 Ligands"
)

In [ ]:
# Example 3: Just pie chart
print("\n\nExample 3: Pie Chart Only")
create_dual_chart_with_arrows(
    data_series=data_ligands['ligand'],
    bar_series=None,
    threshold_percent=2,
    pie_title="Ligand Distribution"
)



In [ ]:
# Test the smart labels version
main_receptors, other_receptors, fig = create_dual_chart_with_arrows(
    data_series=data_ligands['receptor'],
    bar_series=None,
    threshold_percent=2,
    pie_title="Receptor Distribution",
)

#plt.savefig('../../results/comparative_analysis/Pie chart Quantification of Receptors within the Study.png', dpi=300, bbox_inches='tight')

#### ligand receptor connection comparison

In [ ]:
def create_basic_connection_plot(data_ligands, top_n=5, max_ligands_per_receptor=5, figsize=(10, 6)):
    """
    Basic connection plot with connections colored by receptor
    """
    # Clean data
    df = data_ligands.copy()
    df['receptor'] = df['receptor'].str.strip()
    df['ligand'] = df['ligand'].str.strip()
    
    # Get top receptors
    top_receptors = df['receptor'].value_counts().head(top_n).index.tolist()
    
    # Get connections
    connections = []
    for receptor in top_receptors:
        # Get ligands for this receptor
        receptor_df = df[df['receptor'] == receptor]
        ligand_counts = receptor_df['ligand'].value_counts()
        
        # Take top ligands
        top_ligands = ligand_counts.head(max_ligands_per_receptor).index.tolist()
        
        for ligand in top_ligands:
            connections.append((receptor, ligand))
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Position receptors (left side, x=0)
    receptor_y_positions = {}
    for i, receptor in enumerate(top_receptors):
        receptor_y_positions[receptor] = (i + 0.5) / len(top_receptors)
    
    # Get all unique ligands from connections
    all_ligands = list(set([ligand for _, ligand in connections]))
    
    # Position ligands (right side, x=1)
    ligand_y_positions = {}
    for i, ligand in enumerate(all_ligands):
        ligand_y_positions[ligand] = (i + 0.5) / len(all_ligands)
    
    # Create color map for receptors
    import matplotlib.cm as cm
    receptor_colors = cm.tab20c(np.linspace(0, 1, len(top_receptors)))
    receptor_color_map = {receptor: receptor_colors[i] for i, receptor in enumerate(top_receptors)}
    
    # Draw connection lines - colored by receptor
    for receptor, ligand in connections:
        y1 = receptor_y_positions[receptor]
        y2 = ligand_y_positions[ligand]
        
        # Draw straight line with receptor color
        ax.plot([0.1, 0.9], [y1, y2], 
               color=receptor_color_map[receptor], 
               alpha=0.7, linewidth=1.5, zorder=1)
    
    # Draw receptor nodes - use same color as connections
    for receptor, y in receptor_y_positions.items():
        ax.scatter(0.1, y, s=200, color=receptor_color_map[receptor], 
                  edgecolor='black', linewidth=2, zorder=10)
        ax.text(0.08, y, receptor, ha='right', va='center',
                fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))
    
    # Draw ligand nodes
    for ligand, y in ligand_y_positions.items():
        ax.scatter(0.9, y, s=150, color='lightgreen', edgecolor='darkgreen',
                  linewidth=2, zorder=10)
        ax.text(0.92, y, ligand, ha='left', va='center',
                fontsize=9,
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.9))
    
    # Configure plot
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    # Add legend with receptor colors
    from matplotlib.lines import Line2D
    legend_elements = []
    
    # Add receptor legend entries
    for receptor in top_receptors:
        legend_elements.append(
            Line2D([0], [0], marker='o', color='w', 
                   markerfacecolor=receptor_color_map[receptor],
                   markeredgecolor='black', markersize=10, 
                   label=f'{receptor}')
        )
    
    # Add ligand entry
    legend_elements.append(
        Line2D([0], [0], marker='o', color='w', 
               markerfacecolor='lightgreen',
               markeredgecolor='darkgreen', markersize=10, 
               label='Ligands')
    )
    
    ax.legend(handles=legend_elements, loc='upper center', 
              bbox_to_anchor=(0.5, -0.05), ncol=3, fontsize=9)
    
    # Add title at the bottom
    ax.text(0.5, -0.2,
            f'Receptor ↔ Ligand Connections\nTop {top_n} Receptors, {len(connections)} Connections',
            ha='center', va='top', fontsize=15, fontweight='bold',
            transform=ax.transAxes)

    plt.tight_layout()
    plt.show()
    
    print(f"✅ Created connection plot with:")
    print(f"   - {len(top_receptors)} receptors")
    print(f"   - {len(all_ligands)} ligands")
    print(f"   - {len(connections)} connections")
    
    return connections

In [ ]:
connections = create_basic_connection_plot(data_ligands, top_n=5, max_ligands_per_receptor=5)

# Show first 10 connections
print("\nFirst 10 connections:")
for i, (receptor, ligand) in enumerate(connections[:10], 1):
    print(f"{i}. {receptor} → {ligand}")

# Visaulization of parameter receptor and ligand

In [ ]:
import pandas as pd

# Clean the data
data_clean = data_ligands.copy()
data_clean['Parameter'] = data_clean['Parameter'].str.replace(' ', ' ').str.strip()
data_clean['receptor'] = data_clean['receptor'].str.replace(' ', ' ').str.strip()
data_clean['ligand'] = data_clean['ligand'].str.strip()

# Get unique parameters
parameters = data_clean['Parameter'].unique()

# Create summary lists
summary_rows = []

for param in parameters:
    param_data = data_clean[data_clean['Parameter'] == param]
    
    # Get top 3 receptors
    top_receptors = param_data['receptor'].value_counts().head(1)
    receptors_str = ", ".join([f"{rec} ({count})" for rec, count in top_receptors.items()])
    
    # Get top 3 ligands
    top_ligands = param_data['ligand'].value_counts().head(1)
    ligands_str = ", ".join([f"{lig} ({count})" for lig, count in top_ligands.items()])
    
    # Count unique receptors and ligands
    n_receptors = param_data['receptor'].nunique()
    n_ligands = param_data['ligand'].nunique()
    total_entries = len(param_data)
    
    summary_rows.append({
        'Parameter': param,
        'Top 3 Receptors': receptors_str,
        'Top 3 Ligands': ligands_str,
        'Total Entries': total_entries,
        'Unique Receptors': n_receptors,
        'Unique Ligands': n_ligands
    })

# Create summary DataFrame
summary_df = pd.DataFrame(summary_rows)

# Display the table
print("Top 3 Most Studied Receptors and Ligands by Parameter")
print("=" * 100)
print(summary_df.to_string(index=False))

In [ ]:
summary_df

# Mean & Data points for BHM vs REM